# 02. Cleaning decisions

`src/denver311/clean.py` turns the raw files into one table (`requests.parquet`), and every
step writes a line to `cleaning_log.json` with the rows it touched. This notebook shows that
log, the checks that run after loading, and the evidence behind the judgment calls.

In [1]:
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize

from denver311 import DUCKDB_PATH, PROCESSED_DIR, RAW_DIR
from denver311 import style
from denver311.style import NAVY, TEAL, BLUE, SLATE, LIGHT, INK

style.apply()
pd.set_option("display.max_columns", 40, "display.width", 160)
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
q = lambda sql: con.sql(sql).df()

In [2]:
log = pd.DataFrame(json.loads((PROCESSED_DIR / "cleaning_log.json").read_text()))
log.style.format({"rows": "{:,}"}).hide(axis="index")

step,rows,decision
load,"3,012,345",Read 7 yearly files; normalized headers across formats.
drop_exact_duplicates,"2,015",Dropped rows identical on every source field.
trim_text,"11,576","Request type text fixed (non-breaking spaces, padding); all text fields trimmed, blank to null."
drop_empty_columns,5,"Dropped columns under 0.1% filled: division, major_area, type, topic, neighborhood."
merge_agency_names,"44,144",Environmental Health (2019 only) merged into Public Health & Environment; stray Accounting and ROWE labels merged.
merge_status_labels,"50,915","Merged typo and synonym status labels (Transfered, Duplicate Case, Spam, Approved)."
drop_unparseable_created,0,Dropped rows with no parseable created timestamp.
flag_closed_before_created,0,"Close earlier than create: kept the row, nulled hours_to_close."
open_cases,"136,857","No close timestamp: kept as open backlog, excluded from time-to-close percentiles."
zero_duration,"1,317,584",Closed at the same second it was created: counted as handled on contact.


## Quality checks (run by `sql/duckdb/02_quality_checks.sql`)

In [3]:
q("SELECT * FROM quality_checks")

,check_name,failing_rows,tolerance,passed
0,request_id is unique,0,0,True
1,created_at is never null,0,0,True
2,created year matches file year,0,0,True
3,created_at within 2019-2025,0,0,True
4,hours_to_close is non-negative,0,0,True
5,closed rows have a duration,0,0,True
6,coordinates inside Denver box,0,0,True
7,council district in 1-11,0,0,True
8,every row has a category,0,0,True
9,geocoded rows matched to a neighborhood (<3% m...,3508,24196,True


## Decision 1: what counts as a field request

A 311 contact can be a question ("when is my trash day?"), a hand-off to another agency, spam,
or a request for work at a place. Speed only means something for the last group, so a
**field request** is a row tied to a Denver location whose status is answered, resolved, or
still open. Transfers, out-of-jurisdiction calls, duplicates, spam, and cancellations are
excluded.

In [4]:
q('''
SELECT status_group,
       COUNT(*) AS rows,
       COUNT(*) FILTER (WHERE has_location) AS with_location,
       COUNT(*) FILTER (WHERE is_field_request) AS field_requests
FROM requests GROUP BY 1 ORDER BY rows DESC
''')

,status_group,rows,with_location,field_requests
0,answered,1623652,494982,494982
1,transferred,685122,106533,0
2,resolved,334336,273436,273436
3,open,138112,47848,47848
4,invalid,122491,72246,0
5,out_of_jurisdiction,106617,21498,0


## Decision 2: request categories

There are about 48,800 distinct request-type strings. Keyword rules assign a category first; if
no rule matches, the owning agency decides (Safety to Public Safety, Community Planning to
Neighborhood Inspection, and so on). Fewer than 100 rows end up as Other.

In [5]:
q('''
SELECT request_category, COUNT(*) AS contacts, COUNT(*) FILTER (WHERE is_field_request) AS field_requests,
       list(request_type ORDER BY n DESC)[1:4] AS top_types
FROM (SELECT request_category, request_type, is_field_request, COUNT(*) OVER (PARTITION BY request_type) AS n FROM requests)
GROUP BY 1 ORDER BY field_requests DESC
''')

,request_category,contacts,field_requests,top_types
0,Streets & Traffic,210908,131740,"[Pothole, Pothole, Pothole, Pothole]"
1,Neighborhood Inspection,177969,123132,"[Weeds/Vegetation Violation, Weeds/Vegetation ..."
2,Public Safety (non-emergency),483803,120808,"[911 NECT KEEP, 911 NECT KEEP, 911 NECT KEEP, ..."
3,Solid Waste,181462,103234,"[Service Schedule, Service Schedule, Service S..."
4,General Information,506583,67494,"[Other, Other, Other, Other]"
5,Licensing & Permits,250690,67345,"[New Business License, New Business License, N..."
6,Encampments,95308,56365,"[Encampment Reporting, Encampment Reporting, E..."
7,Parking & Vehicles,155394,53198,"[Sheriff - Abandoned vehicle, Sheriff - Abando..."
8,Public Health & Noise,69615,34663,"[OOJ, OOJ, OOJ, OOJ]"
9,Animals,112153,22268,"[Animal Adoption/Hold Inquiry, Animal Adoption..."


## Decision 3: neighborhoods come from coordinates

The source `Neighborhood` column is empty, so each geocoded request is placed in one of the 78
statistical neighborhoods with a point-in-polygon join (DuckDB `spatial`, `ST_Contains`). The
polygons come with ACS 2019-2023 population, which makes per-resident rates possible.

In [6]:
q('''
SELECT COUNT(*) FILTER (WHERE latitude IS NOT NULL) AS geocoded,
       COUNT(*) FILTER (WHERE nbhd_id IS NOT NULL) AS matched,
       round(COUNT(*) FILTER (WHERE nbhd_id IS NOT NULL) / COUNT(*) FILTER (WHERE latitude IS NOT NULL), 4) AS match_rate,
       (SELECT SUM(population) FROM neighborhoods) AS acs_population
FROM requests
''')

,geocoded,matched,match_rate,acs_population
0,806520,803012,0.9957,713734.0


## Decision 4: compare like with like across years

Solid-waste request types (`SWM - ...`) make up tens of thousands of contacts in 2019-2020 and
then almost disappear from the 311 file. Demand did not vanish; the requests stopped flowing
through this dataset. Year-over-year volume therefore uses a like-for-like basket without them,
and speed trends only use request types present in all seven years.

In [7]:
q("SELECT year, field_requests, field_requests_like_for_like, swm_contacts FROM agg_yearly")

,year,field_requests,field_requests_like_for_like,swm_contacts
0,2019,158617,100745,75873
1,2020,122002,88178,37965
2,2021,84127,84118,10
3,2022,107378,107376,2
4,2023,131809,131807,4
5,2024,106496,106496,0
6,2025,105837,105837,0


## Decision 5: flag closure records that cannot be read as speed

A type-year is flagged as a **record gap** when more than 20% of its field requests were never
closed in the published file, or more than 25% of its closures were stamped on a single day.
Either pattern means the timestamp describes record keeping, not service.

In [8]:
q('''
SELECT request_type, year, round(open_in_file_share, 2) AS open_in_file,
       top_close_day, round(top_close_day_share, 2) AS share_closed_that_day
FROM agg_closure_integrity WHERE record_gap ORDER BY request_type, year
''')

,request_type,year,open_in_file,top_close_day,share_closed_that_day
0,Barking Dog,2019,0.85,2019-08-30,0.04
1,Barking Dog,2020,0.84,2020-04-23,0.03
2,Barking Dog,2021,0.86,2021-01-11,0.05
3,Barking Dog,2022,0.88,2022-10-11,0.06
4,Inspection,2025,0.39,2025-11-18,0.03
5,Signal Timing,2024,0.21,2025-01-14,0.13
6,Snow Removal Complaint/Sidewalk,2024,0.92,2024-01-10,0.13
7,Snow Removal Complaint/Sidewalk,2025,0.00,2025-05-06,0.83
8,Trash in Yard Violation,2023,0.55,2023-03-21,0.03
9,Trash in Yard Violation,2024,0.86,2024-04-24,0.05


13 of 175 type-years are flagged. The pattern in code enforcement is striking: weeds, sidewalk
snow, and trash-in-yard cases from 2023 and 2024 sit at `Routed to Agency` with no close
timestamp, then 83% of the 2025 sidewalk snow closures land on one day, May 6, 2025.

**Next:** `03_eda.ipynb`.